# Tests: `fasterai.prune.pruner` (source `nbs/prune/pruner.ipynb`)

In [ ]:
from fastcore.test import *
import torch
import torch.nn as nn
from torch.nn.utils import parametrize
from fasterai.core.criteria import large_final
from fasterai.core.parametrize import _unparametrize
from fasterai.prune.pruner import *

In [ ]:
from fastcore.test import *
import warnings

def _test_model():
    return nn.Sequential(
        nn.Conv2d(3, 16, 3, padding=1),
        nn.BatchNorm2d(16),
        nn.ReLU(),
        nn.Conv2d(16, 32, 3, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Linear(32, 10)
    )

x = torch.randn(1, 3, 8, 8)

# A fraction means what it says: 0.4 removes ~40% of the filters
model = _test_model()
pruner = Pruner(model, 0.4, 'local', large_final, example_inputs=x)
params_before = sum(p.numel() for p in model.parameters())
pruner.prune_model()
test_close(model[0].out_channels / 16, 0.6, eps=0.05)
test_close(model[3].out_channels / 32, 0.6, eps=0.05)
assert sum(p.numel() for p in model.parameters()) < params_before

# Model still produces valid output after pruning
out = model(x)
test_eq(out.shape[0], 1)
test_eq(out.shape[1], 10)

# The ratio is stored as the fraction it was given, and handed to torch-pruning unscaled
test_eq(pruner.pruning_ratio, 0.4)
test_eq(pruner.default_pruning_ratio, 0.4)

# A percent is read as x/100 for one release, warns, and prunes exactly like the fraction
_pm = _test_model()
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    _pp = Pruner(_pm, 40, 'local', large_final, example_inputs=x)
_pp.prune_model()
test_eq(_pp.pruning_ratio, 0.4)
test_eq(_pm[0].out_channels, model[0].out_channels)
test_eq({_x.category for _x in w}, {FutureWarning})

# Nothing to prune, out of range, or not a number
with ExceptionExpected(ValueError, regex='removes nothing'):
    Pruner(_test_model(), 0, 'local', large_final, example_inputs=x)
with ExceptionExpected(ValueError, regex='fraction'):
    Pruner(_test_model(), 150, 'local', large_final, example_inputs=x)
with ExceptionExpected(TypeError, regex='must be a number'):
    Pruner(_test_model(), '0.4', 'local', large_final, example_inputs=x)

# Per-layer dict: 0 leaves a layer alone
_dm = _test_model()
_dp = Pruner(_dm, {'0': 0.3, '3': 0}, 'local', large_final, example_inputs=x)
_dp.prune_model()
assert _dm[0].out_channels < 16, _dm[0].out_channels
test_eq(_dm[3].out_channels, 32)
test_eq(_dp.pruning_ratio, {'0': 0.3, '3': 0.0})

# An explicit 0 survives a non-zero default_pruning_ratio
_dm2 = _test_model()
_dp2 = Pruner(_dm2, {'0': 0.3, '3': 0}, 'local', large_final, example_inputs=x,
              default_pruning_ratio=0.5)
_dp2.prune_model()
test_eq(_dp2.default_pruning_ratio, 0.5)
assert _dm2[0].out_channels < 16, _dm2[0].out_channels
test_eq(_dm2[3].out_channels, 32)

# A bad per-layer value names its layer
with ExceptionExpected(ValueError, regex="'3'"):
    Pruner(_test_model(), {'0': 0.3, '3': 150}, 'local', large_final, example_inputs=x)

# A model whose parameters are all frozen traces nothing, so pruning would silently do nothing
_fm = _test_model()
_fm.requires_grad_(False)
with ExceptionExpected(ValueError, regex='requires_grad_'):
    Pruner(_fm, 0.4, 'local', large_final, example_inputs=x)

# ... and it prunes once the parameters require grad again
_fm.requires_grad_(True)
Pruner(_fm, 0.4, 'local', large_final, example_inputs=x).prune_model()
assert _fm[0].out_channels < 16, _fm[0].out_channels

# A tensor subclass (a fastai `TensorImage`, say) is traced as a plain tensor: without that,
# torch-pruning misses the batch-norm that follows a pruned convolution
class _MyTensor(torch.Tensor): pass

class _Residual(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1, self.bn1 = nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16)
        self.conv2, self.bn2 = nn.Conv2d(16, 16, 3, padding=1), nn.BatchNorm2d(16)
        self.pool, self.fc = nn.AdaptiveAvgPool2d(1), nn.Linear(16, 10)
    def forward(self, x):
        x = torch.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(x)); y += x
        return self.fc(self.pool(torch.relu(y)).flatten(1))

_rm = _Residual()
Pruner(_rm, 0.4, 'local', large_final, example_inputs=x.as_subclass(_MyTensor)).prune_model()
assert _rm.conv2.out_channels < 16, _rm.conv2.out_channels
test_eq(_rm.bn2.running_mean.numel(), _rm.conv2.out_channels)
test_eq(_rm(x).shape, (1, 10))

In [ ]:
# --- a model whose weight is computed by a parametrization is refused, beside the frozen refusal ---
# torch-pruning rewrites the modules it traces, which a parametrization would leave behind.
class _Double(nn.Module):
    "A parametrization with a visible effect: the weight this module computes is twice its master"
    def forward(self, w): return w * 2

_pm = _test_model()
parametrize.register_parametrization(_pm[3], 'weight', _Double())
with ExceptionExpected(ValueError, regex='bake or strip it before pruning'):
    Pruner(_pm, 0.4, 'local', large_final, example_inputs=x)
with ExceptionExpected(ValueError, regex="'3'"):   # ...naming the module that carries it
    Pruner(_pm, 0.4, 'local', large_final, example_inputs=x)

# ...and it prunes once the parametrization is gone
_unparametrize(_pm[3])
Pruner(_pm, 0.4, 'local', large_final, example_inputs=x).prune_model()
assert _pm[3].out_channels < 32, _pm[3].out_channels
test_eq(_pm(x).shape, (1, 10))

In [ ]:
# --- group sparse learning: `reg > 0` adds DepGraph's group penalty to the gradients ---
import torch_pruning as tp

class _Block(nn.Module):
    "conv1's channels are coupled through a residual add to convB, convA's through convB's input"
    def __init__(self, mid=6, dw=False):
        super().__init__()
        self.conv1, self.bn1 = nn.Conv2d(3, 8, 3, padding=1, bias=False), nn.BatchNorm2d(8)
        self.convA, self.bnA = nn.Conv2d(8, mid, 1, bias=False), nn.BatchNorm2d(mid)
        self.dw = nn.Conv2d(mid, mid, 3, padding=1, groups=mid, bias=False) if dw else nn.Identity()
        self.convB, self.bnB = nn.Conv2d(mid, 8, 1, bias=False), nn.BatchNorm2d(8)
        self.fc = nn.Linear(8, 4)
        for bn in (self.bn1, self.bnA, self.bnB): nn.init.uniform_(bn.weight, 0.5, 1.5)
    def forward(self, x):
        x = torch.relu(self.bn1(self.conv1(x)))
        y = self.bnB(self.convB(self.dw(torch.relu(self.bnA(self.convA(x))))))
        return self.fc((x + y).mean((2, 3)))

def _block(ratio=0.5, seed=0, **kw):
    torch.manual_seed(seed); m = _Block(**{k: kw.pop(k) for k in ('mid', 'dw') if k in kw})
    return m, Pruner(m, ratio, 'local', large_final, example_inputs=x, **kw)

def _grads(m):
    m.zero_grad(); m(x).pow(2).sum().backward()
    return {n: p.grad.clone() for n, p in m.named_parameters() if p.grad is not None}

def _delta(m, p, **kw):
    "What `regularize` adds to every gradient"
    g0 = _grads(m); p.regularize(**kw)
    return {n: q.grad - g0[n] for n, q in m.named_parameters() if n in g0}

def _gamma(p, root, alpha):
    "Per-channel penalty factor of the group that prunes `root`'s filters, from the pruner's own group importance"
    g = next(g for g in p.pruner._groups if any(d.target.module is root and p.pruner.DG.is_out_channel_pruning_fn(d.handler) for d, _ in g))
    imp = p.group_importance(g).sqrt()
    return (2**alpha) ** ((imp.max() - imp) / (imp.max() - imp.min()))

# one group computed by hand: filters of |w| 1 and 4, fc columns of 1 -> importance (1, 2.5), so gamma = (2**3, 1)
class _Tiny(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv, self.bn, self.fc = nn.Conv2d(1, 2, 1, bias=False), nn.BatchNorm2d(2), nn.Linear(2, 1)
    def forward(self, x): return self.fc(self.bn(self.conv(x)).mean((2, 3)))
tm = _Tiny()
with torch.no_grad(): tm.conv.weight.copy_(torch.tensor([1., 4.]).view(2, 1, 1, 1)); tm.fc.weight.fill_(1.)
tpr = Pruner(tm, 0.5, 'local', large_final, example_inputs=torch.randn(2, 1, 2, 2), reg=0.01, alpha=3)
for q in tm.parameters(): q.grad = torch.zeros_like(q)
tpr.regularize()
test_close(tm.conv.weight.grad.flatten(), torch.tensor([0.08, 0.04]), eps=1e-7)   # 0.01*8*1, 0.01*1*4
test_close(tm.bn.weight.grad, torch.tensor([0.08, 0.01]), eps=1e-7)
test_close(tm.fc.weight.grad, torch.tensor([[0.08, 0.01]]), eps=1e-7)

reg, alpha = 1e-2, 3
m, p = _block(reg=reg, alpha=alpha)
assert isinstance(p.pruner, tp.pruner.GroupNormPruner)
test_eq(p.group_importance(p.pruner._groups[0]).device, m.conv1.weight.device)   # on the model's device, not fastai's default
g1, gA = _gamma(p, m.conv1, alpha), _gamma(p, m.convA, alpha)
d = _delta(m, p)
v = lambda g, dim, nd: g.view([-1 if i == dim else 1 for i in range(nd)])
# out-channels, the BN weights and the dependent in-channels of both groups, on every coupled tensor
test_close(d['conv1.weight'], reg * v(g1, 0, 4) * m.conv1.weight, eps=1e-7)
test_close(d['bn1.weight'],   reg * g1 * m.bn1.weight, eps=1e-7)
test_close(d['bnB.weight'],   reg * g1 * m.bnB.weight, eps=1e-7)
test_close(d['bnA.weight'],   reg * gA * m.bnA.weight, eps=1e-7)
test_close(d['convA.weight'], reg * (v(gA, 0, 4) + v(g1, 1, 4)) * m.convA.weight, eps=1e-7)
test_close(d['convB.weight'], reg * (v(g1, 0, 4) + v(gA, 1, 4)) * m.convB.weight, eps=1e-7)
test_close(d['fc.weight'],    reg * v(g1, 1, 2) * m.fc.weight, eps=1e-7)   # the output layer is kept, its input columns follow the group
for n in ('bn1.bias', 'bnA.bias', 'bnB.bias', 'fc.bias'): test_eq(d[n].abs().max().item(), 0.)

# `scale` multiplies the penalty (for scaled gradients: AMP, accumulation)
test_close(_delta(m, p, scale=3.)['convB.weight'], 3 * d['convB.weight'], eps=1e-7)

In [ ]:
# --- reg=0 builds the same MetaPruner as before and has nothing to apply ---
m0, p0 = _block()
test_is(type(p0.pruner), tp.pruner.MetaPruner)
with ExceptionExpected(ValueError, regex='reg > 0'): p0.regularize()
with ExceptionExpected(ValueError, regex='reg must be >= 0'): _block(reg=-1e-4)

# the penalty lives in the gradients only: without regularize(), reg>0 prunes exactly like reg=0
m, p = _block(reg=reg); p.prune_model(); p0.prune_model()
for (n, a), b in zip(m.state_dict().items(), m0.state_dict().values()): test_eq(a, b)

In [ ]:
# --- after a prune the groups are rebuilt on the smaller layers ---
d = _delta(m, p)
test_eq(m.bn1.weight.numel(), 4)
test_close(d['bn1.weight'], reg * _gamma(p, m.conv1, 4) * m.bn1.weight, eps=1e-7)
test_close(d['bnA.weight'], reg * _gamma(p, m.convA, 4) * m.bnA.weight, eps=1e-7)

In [ ]:
# --- only the groups the pruner will prune are penalized ---
m, p = _block(ratio={'convA': 0.5}, reg=reg)
d = _delta(m, p)
for n in ('conv1.weight', 'bn1.weight', 'bnB.weight', 'fc.weight'): test_eq(d[n].abs().max().item(), 0.)
test_close(d['bnA.weight'], reg * _gamma(p, m.convA, 4) * m.bnA.weight, eps=1e-7)
test_close(d['convB.weight'], reg * v(_gamma(p, m.convA, 4), 1, 4) * m.convB.weight, eps=1e-7)
m, p = _block(ratio={'convA': 0.}, default_pruning_ratio=0.5, reg=reg)   # a 0 in the dict leaves that layer alone
d = _delta(m, p)
test_eq(d['bnA.weight'].abs().max().item(), 0.)
assert d['bn1.weight'].abs().min() > 0

# a frozen layer gets no penalty
m, p = _block(reg=reg)
m.bn1.requires_grad_(False)
d = _delta(m, p)
assert 'bn1.weight' not in d and m.bn1.weight.grad is None
assert d['bnB.weight'].abs().min() > 0

# a group with a single channel, or a constant importance, is skipped instead of turning the gradients into nan
m, p = _block(mid=1, reg=reg)
d = _delta(m, p)
assert all(t.isfinite().all() for t in d.values())
test_eq(d['bnA.weight'].abs().max().item(), 0.)
assert d['bn1.weight'].abs().min() > 0
m, p = _block(reg=reg)
for c in (m.conv1, m.convA, m.convB, m.fc): nn.init.constant_(c.weight, 0.1)
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    d = _delta(m, p)
test_eq(len(w), 1); assert 'no penalty was applied' in str(w[0].message)
assert all(t.abs().max() == 0 for t in d.values())

# depthwise: the group around it is penalized, its own kernels are not (they are not scored either)
m, p = _block(dw=True, reg=reg)
d = _delta(m, p)
test_eq(d['dw.weight'].abs().max().item(), 0.)
test_close(d['bnA.weight'], reg * _gamma(p, m.convA, 4) * m.bnA.weight, eps=1e-7)
p.prune_model(); _delta(m, p)
test_eq(m.dw.weight.shape[0], m.bnA.weight.numel())

In [ ]:
# --- on a real model: every penalized gradient stays finite, before and after a prune ---
from torchvision.models import resnet18
torch.manual_seed(0); m = resnet18(num_classes=10)
p = Pruner(m, 0.3, 'local', large_final, iterative_steps=2, example_inputs=torch.randn(1, 3, 32, 32), reg=1e-3)
for _ in range(2):
    xb = torch.randn(2, 3, 32, 32)
    m.zero_grad(); m(xb).sum().backward()
    g0 = {n: q.grad.clone() for n, q in m.named_parameters()}
    p.regularize()
    assert all(q.grad.isfinite().all() for q in m.parameters())
    changed = [n for n, q in m.named_parameters() if not torch.equal(q.grad, g0[n])]
    assert len(changed) > 10 and not any(n.endswith('bias') for n in changed), changed
    p.prune_model()
test_eq(m(xb).shape, (2, 10))